## Transform II: Überschwemmungen transformieren

In diesem Abschnitt werden alle geforderten Transformationsschritte umgesetzt, aber die beiden Datensätze werden noch nicht miteinander verbunden. EMDAT wird fachlich auf Überschwemmungen im Mittelmeerraum vorbereitet, Sea-Level wird separat als Zeitreihe vorbereitet. Die eigentliche Zusammenführung der Datensätze passiert erst später.

**Was hier passiert:** Die Transformationslogik baut auf den Pipeline-Funktionen `run_import`, `run_transform` und `run_cleaning` auf. Danach werden daraus analysefertige Flood- und Sea-Level-Tabellen erzeugt, inklusive abgeleiteter Variablen, Aggregationen, Long/Wide-Umformungen, Zeitreihenfeatures, Skalierungen und Logging-Beispielen.


### Entscheid: Long oder Wide? To do: weiter unten anlegen

Bevor die Daten transformiert werden, wird für jede zentrale Zwischentabelle entschieden, ob ein Long- oder Wide-Format sinnvoller ist. Diese Entscheidung ist wichtig, weil Gruppierungen, Visualisierungen und Modellfeatures unterschiedliche Tabellenformen brauchen.

**Input:** geplante Zwischentabellen aus EMDAT-Floods und Sea-Level.

**Output:** eine begründete Übersicht, welche Tabelle in welchem Format weiterverwendet wird.

Die Spalten der Tabelle bedeuten: `Datensatz / Zwischentabelle` benennt die jeweilige Tabelle, `Gewähltes Format` hält die Formatentscheidung fest und `Grund` erklärt, warum dieses Format für den nächsten Arbeitsschritt sinnvoll ist.

| Datensatz / Zwischentabelle | Gewähltes Format | Grund |
|---|---|---|
| `emdat_flood` | **Wide** | Eine Zeile ist ein Flood-Ereignis; Spalten enthalten Land, Datum, Subtyp und Impact. |
| `flood_subtype_long` | **Long** | Gut für Gruppierungen und Visualisierungen nach Flood-Subtyp. |
| `flood_subtype_wide` | **Wide** | Gut für spätere Modellfeatures, weil jeder Subtyp eine eigene Spalte hat. |
| `sea_level_ts` | **Long/Zeitreihe** | Eine Zeile pro Zeitpunkt ist ideal für Resampling, Lag/Lead und Rolling Windows. |
| `sea_metric_long` | **Long** | Mehrere Sea-Level-Metriken werden untereinander gestellt, ohne mit EMDAT zu joinen. |


In [ ]:
# Für unsere analyse ist das nicht relevant

import logging

import numpy as np
import pandas as pd

from myproj.pipeline import run_import, run_transform, run_cleaning

pd.set_option("display.max_columns", 60)


def show_table_info(title, column_descriptions):
    """Show a title and compact column explanation before a displayed table."""
    print(f"\n{title}")
    display(pd.DataFrame(column_descriptions, columns=["Spalte", "Erklärung"]))

emdat_raw, sea_level_raw = run_import()

emdat, sea_level = run_transform(emdat_raw, sea_level_raw)
emdat_clean, sea_level_clean = run_cleaning(emdat, sea_level)

# Ab hier wird mit den bereinigten Pipeline-Outputs weitergearbeitet.
emdat = emdat_clean
sea_level = sea_level_clean

print(f"EMDAT Setup nach Transform/Cleaning: {emdat.shape[0]:,} Zeilen, {emdat.shape[1]} Spalten")
print(f"Sea-Level Setup nach Transform/Cleaning: {sea_level.shape[0]:,} Zeilen, {sea_level.shape[1]} Spalten")

SEA_VALUE = "MSL_filtered_GIA_corrected_adjusted"
SEA_TREND = "trend_MSL_filtered_GIA_corrected_adjusted"

MED_ISO = [
    "ALB", "DZA", "BIH", "HRV", "CYP", "EGY", "FRA", "GRC", "ISR", "ITA", "LBN",
    "LBY", "MLT", "MNE", "MAR", "PSE", "SVN", "ESP", "SYR", "TUN", "TUR",
]

def make_event_date(df, prefix):
    """Create a date from EMDAT year/month/day columns and keep missingness transparent."""
    return pd.to_datetime(
        pd.DataFrame(
            {
                "year": pd.to_numeric(df[f"{prefix} Year"], errors="coerce"),
                "month": pd.to_numeric(df[f"{prefix} Month"], errors="coerce").fillna(1),
                "day": pd.to_numeric(df[f"{prefix} Day"], errors="coerce").fillna(1),
            }
        ),
        errors="coerce",
    )

def date_quality(df, prefix):
    month_missing = df[f"{prefix} Month"].isna()
    day_missing = df[f"{prefix} Day"].isna()
    return np.select(
        [month_missing & day_missing, month_missing, day_missing],
        ["month_and_day_imputed", "month_imputed", "day_imputed"],
        default="complete",
    )

def season_from_month(month):
    if pd.isna(month):
        return "unknown"
    month = int(month)
    if month in [12, 1, 2]:
        return "winter"
    if month in [3, 4, 5]:
        return "spring"
    if month in [6, 7, 8]:
        return "summer"
    return "autumn"

def standard_scale(series): # raus
    x = pd.to_numeric(series, errors="coerce").astype(float)
    std = x.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(np.nan, index=x.index)
    return (x - x.mean()) / std

def minmax_scale(series): # raus
    x = pd.to_numeric(series, errors="coerce").astype(float)
    value_range = x.max() - x.min()
    if value_range == 0 or pd.isna(value_range):
        return pd.Series(np.nan, index=x.index)
    return (x - x.min()) / value_range

def box_cox_transform(series, lmbda=0.0): # raus
    """Box-Cox needs positive values; shifting by min + 1 preserves ordering."""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    x_positive = x - x.min(skipna=True) + 1
    if np.isclose(lmbda, 0):
        transformed = np.log(x_positive)
    else:
        transformed = (np.power(x_positive, lmbda) - 1) / lmbda
    return pd.Series(transformed, index=x.index)

def yeo_johnson_transform(series, lmbda=0.5): # raus
    """Yeo-Johnson also works with zero and negative values."""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    transformed = pd.Series(np.nan, index=x.index, dtype="float64")
    valid = x.notna()
    pos = valid & (x >= 0)
    neg = valid & (x < 0)

    if np.isclose(lmbda, 0):
        transformed.loc[pos] = np.log1p(x.loc[pos])
    else:
        transformed.loc[pos] = (np.power(x.loc[pos] + 1, lmbda) - 1) / lmbda

    if np.isclose(lmbda, 2):
        transformed.loc[neg] = -np.log1p(-x.loc[neg])
    else:
        transformed.loc[neg] = -(
            (np.power(-x.loc[neg] + 1, 2 - lmbda) - 1) / (2 - lmbda)
        )
    return transformed


2026-04-29 12:51:28 | myproj.pipeline      | INFO     | run_import | start
2026-04-29 12:51:28 | myproj.io            | INFO     | Lade Rohdatei: public_emdat_1991_2024.xlsx
2026-04-29 12:51:32 | myproj.io            | INFO     | load_raw_data | file: public_emdat_1991_2024.xlsx | rows: 20657 | cols: 47
2026-04-29 12:51:32 | myproj.io            | INFO     | Lade Rohdatei: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc
2026-04-29 12:51:32 | myproj.io            | INFO     | load_raw_data | file: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc | variables: 2 | dimensions: {'time': 9405}
2026-04-29 12:51:32 | myproj.pipeline      | INFO     | run_import | done
2026-04-29 12:51:32 | myproj.pipeline      | INFO     | run_transform | start
2026-04-29 12:51:32 | myproj.transform     | INFO     | select_columns | cols: 47 → 18
2026-04-29 12:51:32 | myproj.transform     | INFO     | filter_before_sea_level_start | rows: 20657 → 16769 | removed: 3888
202

EMDAT Setup: 16,769 Zeilen, 18 Spalten
Sea-Level Setup: 9,405 Zeilen, 3 Spalten


### EMDAT Floods: filtern, sortieren, kombinieren, Strings und abgeleitete Variablen

In diesem Abschnitt wird der transformierte EMDAT-Datensatz auf relevante Überschwemmungen im Mittelmeerraum eingeschränkt. Zusätzlich werden Datumsvariablen, Qualitätsmarker, bereinigte Textspalten und Impact-Variablen erzeugt.

**Input:** `emdat` nach `run_transform()` und `run_cleaning()` und die Sea-Level-Zeitabdeckung aus `sea_level`.

**Was gemacht wird:** Es werden Flood-Ereignisse erkannt, Mittelmeer-Länder gefiltert, ungültige oder nicht abgedeckte Zeitpunkte ausgeschlossen, Ereignisse sortiert und Teile der Ereignis-ID mit Regex extrahiert.

**Output:** `emdat_flood`, eine ereignisbasierte Wide-Tabelle mit bereinigten und abgeleiteten Variablen.


In [ ]:
emdat_transform = emdat.copy()
emdat_transform["start_date"] = make_event_date(emdat_transform, "Start")
emdat_transform["end_date"] = make_event_date(emdat_transform, "End")
emdat_transform["start_date_quality"] = date_quality(emdat_transform, "Start")
emdat_transform["end_date_quality"] = date_quality(emdat_transform, "End")
emdat_transform["year_month"] = emdat_transform["start_date"].dt.to_period("M").dt.to_timestamp() # nicht monatlich sondern täglich aggregieren
emdat_transform["season"] = emdat_transform["Start Month"].apply(season_from_month)
emdat_transform["event_duration_days"] = (
    (emdat_transform["end_date"] - emdat_transform["start_date"]).dt.days + 1
).clip(lower=1)
emdat_transform["has_coordinates"] = emdat_transform[["Latitude", "Longitude"]].notna().all(axis=1)
emdat_transform["affected_total_filled"] = pd.to_numeric(
    emdat_transform["Total Affected"], errors="coerce"
).fillna(0)
emdat_transform["deaths_total_filled"] = pd.to_numeric(
    emdat_transform["Total Deaths"], errors="coerce"
).fillna(0)
emdat_transform["impact_total"] = emdat_transform["affected_total_filled"] + emdat_transform["deaths_total_filled"]
emdat_transform["impact_log1p"] = np.log1p(emdat_transform["impact_total"])

sea_coverage_time = pd.to_datetime(sea_level["time"])
sea_coverage_start = sea_coverage_time.min()
sea_coverage_end = sea_coverage_time.max()

flood_text_mask = (
    emdat_transform["Disaster Type"].str.contains("flood", case=False, na=False)
    | emdat_transform["Disaster Subtype"].str.contains("flood", case=False, na=False)
)
mediterranean_mask = emdat_transform["ISO"].isin(MED_ISO)
has_valid_date_mask = emdat_transform["start_date"].notna()
within_sea_coverage_mask = emdat_transform["start_date"].between(sea_coverage_start, sea_coverage_end)

emdat_flood = (
    emdat_transform.loc[
        flood_text_mask & mediterranean_mask & has_valid_date_mask & within_sea_coverage_mask
    ]
    .copy()
    .sort_values(["start_date", "Country", "DisNo."])
    .reset_index(drop=True)
)

emdat_flood["country_clean"] = emdat_flood["Country"].str.replace(r"\s+", " ", regex=True).str.strip()
emdat_flood["flood_subtype_clean"] = (
    emdat_flood["Disaster Subtype"]
    .fillna("unknown")
    .str.lower()
    .str.replace(r"\s*\([^)]*\)", "", regex=True)
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
emdat_flood["origin_clean"] = (
    emdat_flood["Origin"]
    .fillna("unknown")
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
emdat_flood["flood_year"] = emdat_flood["start_date"].dt.year
emdat_flood["flood_month"] = emdat_flood["start_date"].dt.month
emdat_flood["affected_per_death"] = np.where(
    emdat_flood["deaths_total_filled"] > 0,
    emdat_flood["affected_total_filled"] / emdat_flood["deaths_total_filled"],
    np.nan,
)

disno_parts = emdat_flood["DisNo."].str.extract(
    r"^(?P<disno_year>\d{4})-(?P<disno_sequence>\d{4})-(?P<disno_iso>[A-Z]{3})$"
)
disno_parts["disno_year"] = pd.to_numeric(disno_parts["disno_year"], errors="coerce")
disno_parts["disno_sequence"] = pd.to_numeric(disno_parts["disno_sequence"], errors="coerce")
emdat_flood = pd.concat([emdat_flood, disno_parts], axis=1)

print(f"Flood-Ereignisse im Mittelmeerraum: {len(emdat_flood):,}")
print(f"Sea-Level-Abdeckung: {sea_coverage_start.date()} bis {sea_coverage_end.date()}")
print(f"Flood-Zeitraum nach Ausrichtung: {emdat_flood['start_date'].min().date()} bis {emdat_flood['start_date'].max().date()}")
show_table_info(
    "Tabelle: EMDAT Flood-Ereignisse im Mittelmeerraum",
    [
        ("DisNo.", "Eindeutige Ereignis-ID aus EMDAT."),
        ("start_date", "Berechnetes Startdatum des Ereignisses."),
        ("Country", "Originaler Ländername aus EMDAT."),
        ("country_clean", "Bereinigter Ländername ohne überflüssige Leerzeichen."),
        ("Disaster Type", "Übergeordneter Katastrophentyp."),
        ("Disaster Subtype", "Detaillierter Flood-Subtyp im Originaltext."),
        ("flood_subtype_clean", "Bereinigter und normalisierter Flood-Subtyp."),
        ("origin_clean", "Bereinigter Ursprung oder Auslöser des Ereignisses."),
        ("impact_total", "Summe aus betroffenen Personen und Todesfällen."),
        ("impact_log1p", "Log-transformierter Impact für weniger Schiefe."),
        ("disno_year", "Aus der Ereignis-ID extrahiertes Jahr."),
        ("disno_sequence", "Aus der Ereignis-ID extrahierte Laufnummer."),
        ("disno_iso", "Aus der Ereignis-ID extrahierter ISO-Ländercode."),
    ],
)
display(
    emdat_flood[
        [
            "DisNo.", "start_date", "Country", "country_clean", "Disaster Type",
            "Disaster Subtype", "flood_subtype_clean", "origin_clean", "impact_total",
            "impact_log1p", "disno_year", "disno_sequence", "disno_iso",
        ]
    ].head(10)
)

Flood-Ereignisse im Mittelmeerraum: 292
Sea-Level-Abdeckung: 1999-02-20 bis 2024-11-19
Flood-Zeitraum nach Ausrichtung: 1999-11-12 bis 2024-11-13

Tabelle: EMDAT Flood-Ereignisse im Mittelmeerraum


,Spalte,Erklärung
0,DisNo.,Eindeutige Ereignis-ID aus EMDAT.
1,start_date,Berechnetes Startdatum des Ereignisses.
2,Country,Originaler Ländername aus EMDAT.
3,country_clean,Bereinigter Ländername ohne überflüssige Leerz...
4,Disaster Type,Übergeordneter Katastrophentyp.
5,Disaster Subtype,Detaillierter Flood-Subtyp im Originaltext.
6,flood_subtype_clean,Bereinigter und normalisierter Flood-Subtyp.
7,origin_clean,Bereinigter Ursprung oder Auslöser des Ereigni...
8,impact_total,Summe aus betroffenen Personen und Todesfällen.
9,impact_log1p,Log-transformierter Impact für weniger Schiefe.


,DisNo.,start_date,Country,country_clean,Disaster Type,Disaster Subtype,flood_subtype_clean,origin_clean,impact_total,impact_log1p,disno_year,disno_sequence,disno_iso
0,1999-0450-FRA,1999-11-12,France,France,Flood,Riverine flood,riverine_flood,brief_torrential_rain,3041.0,8.020270,1999,450,FRA
1,2000-0281-TUR,2000-05-27,Türkiye,Türkiye,Flood,Riverine flood,riverine_flood,heavy_rains,1002.0,6.910751,2000,281,TUR
2,2000-0356-FRA,2000-06-09,France,France,Flood,Riverine flood,riverine_flood,heavy_rainfall_and_snowmelt,201.0,5.308268,2000,356,FRA
3,2000-0359-ESP,2000-06-10,Spain,Spain,Flood,Flash flood,flash_flood,extreme_rain,516.0,6.248043,2000,359,ESP
4,2000-0775-FRA,2000-07-06,France,France,Flood,Riverine flood,riverine_flood,heavy_rain,601.0,6.400257,2000,775,FRA
5,2000-0577-ITA,2000-09-08,Italy,Italy,Flood,Riverine flood,riverine_flood,heavy_rain,735.0,6.601230,2000,577,ITA
6,2000-0632-ITA,2000-09-20,Italy,Italy,Flood,Flood (General),flood,unknown,1000.0,6.908755,2000,632,ITA
7,2000-0671-ITA,2000-10-13,Italy,Italy,Flood,Riverine flood,riverine_flood,brief_torrential_rain,43025.0,10.669560,2000,671,ITA
8,2000-0699-ESP,2000-10-20,Spain,Spain,Flood,Flash flood,flash_flood,brief_torrential_rain,8010.0,8.988571,2000,699,ESP
9,2000-0692-DZA,2000-10-22,Algeria,Algeria,Flood,Flash flood,flash_flood,brief_torrential_rain,128.0,4.859812,2000,692,DZA


### EMDAT Floods: gruppieren, aggregieren und Long/Wide umwandeln

Hier werden die einzelnen Flood-Ereignisse zu monatlichen Kennzahlen verdichtet. Zusätzlich wird gezeigt, wie eine Subtyp-Tabelle zuerst vom Long- ins Wide-Format und danach wieder zurück ins Long-Format umgewandelt wird.

**Input:** `emdat_flood` aus dem vorherigen Abschnitt.

**Was gemacht wird:** Flood-Subtypen werden pro Monat gezählt, mit `pivot_table()` in Feature-Spalten umgewandelt, mit `melt()` zurücktransformiert und zusätzlich monatlich sowie nach Land/Jahr aggregiert.

**Output:** `flood_monthly`, `flood_by_country_year`, `flood_subtype_long`, `flood_subtype_wide` und `flood_subtype_long_again`.


In [15]:
flood_subtype_long = (
    emdat_flood.groupby(["year_month", "flood_subtype_clean"])
    .size()
    .reset_index(name="n_flood_events")
)

flood_subtype_wide = (
    flood_subtype_long.pivot_table(
        index="year_month",
        columns="flood_subtype_clean",
        values="n_flood_events",
        fill_value=0,
    )
    .add_prefix("n_flood_subtype_")
    .reset_index()
)
flood_subtype_wide.columns.name = None

flood_subtype_long_again = flood_subtype_wide.melt(
    id_vars="year_month",
    var_name="flood_subtype",
    value_name="n_flood_events",
)
flood_subtype_long_again["flood_subtype"] = flood_subtype_long_again["flood_subtype"].str.replace(
    r"^n_flood_subtype_", "", regex=True
)

flood_monthly_base = (
    emdat_flood.groupby("year_month")
    .agg(
        flood_events=("DisNo.", "nunique"),
        flood_countries=("ISO", "nunique"),
        flood_affected_total=("affected_total_filled", "sum"),
        flood_deaths_total=("deaths_total_filled", "sum"),
        flood_median_duration_days=("event_duration_days", "median"),
        flood_coordinates_share=("has_coordinates", "mean"),
        flood_primary_subtype=("flood_subtype_clean", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
    )
    .reset_index()
)

# Kombination innerhalb des EMDAT-Flood-Datensatzes, kein Join mit Sea-Level.
flood_monthly = flood_monthly_base.merge(flood_subtype_wide, on="year_month", how="left")
flood_subtype_cols = [c for c in flood_monthly.columns if c.startswith("n_flood_subtype_")]
flood_monthly[flood_subtype_cols] = flood_monthly[flood_subtype_cols].fillna(0).astype(int)
flood_monthly["affected_per_flood"] = np.where(
    flood_monthly["flood_events"] > 0,
    flood_monthly["flood_affected_total"] / flood_monthly["flood_events"],
    0,
)
flood_monthly["flood_death_share"] = np.where(
    flood_monthly["flood_affected_total"] > 0,
    flood_monthly["flood_deaths_total"] / flood_monthly["flood_affected_total"],
    0,
)

flood_by_country_year = (
    emdat_flood.groupby(["flood_year", "country_clean"])
    .agg(
        flood_events=("DisNo.", "nunique"),
        affected_total=("affected_total_filled", "sum"),
        deaths_total=("deaths_total_filled", "sum"),
    )
    .reset_index()
    .sort_values(["flood_year", "flood_events", "affected_total"], ascending=[True, False, False])
)

print(f"Flood-Monatsaggregation: {flood_monthly.shape[0]:,} Monate mit Floods")
print(f"Long -> Wide -> Long: {flood_subtype_long.shape} -> {flood_subtype_wide.shape} -> {flood_subtype_long_again.shape}")
show_table_info(
    "Tabelle: Monatliche Flood-Aggregation",
    [
        ("year_month", "Monat der Aggregation."),
        ("flood_events", "Anzahl eindeutiger Flood-Ereignisse pro Monat."),
        ("flood_countries", "Anzahl betroffener Länder pro Monat."),
        ("flood_affected_total", "Summe betroffener Personen pro Monat."),
        ("flood_deaths_total", "Summe der Todesfälle pro Monat."),
        ("flood_median_duration_days", "Median der Ereignisdauer in Tagen."),
        ("flood_coordinates_share", "Anteil Ereignisse mit vorhandenen Koordinaten."),
        ("flood_primary_subtype", "Häufigster Flood-Subtyp im Monat."),
        ("n_flood_subtype_*", "Anzahl Ereignisse je Flood-Subtyp als Wide-Feature."),
        ("affected_per_flood", "Durchschnittlich betroffene Personen pro Flood-Ereignis."),
        ("flood_death_share", "Verhältnis Todesfälle zu betroffenen Personen."),
    ],
)
display(flood_monthly.head())
show_table_info(
    "Tabelle: Floods nach Land und Jahr",
    [
        ("flood_year", "Jahr des Flood-Ereignisses."),
        ("country_clean", "Bereinigter Ländername."),
        ("flood_events", "Anzahl eindeutiger Flood-Ereignisse pro Land und Jahr."),
        ("affected_total", "Summe betroffener Personen pro Land und Jahr."),
        ("deaths_total", "Summe Todesfälle pro Land und Jahr."),
    ],
)
display(flood_by_country_year.head(10))
show_table_info(
    "Tabelle: Flood-Subtypen wieder im Long-Format",
    [
        ("year_month", "Monat der Aggregation."),
        ("flood_subtype", "Bereinigter Flood-Subtyp nach Rücktransformation ins Long-Format."),
        ("n_flood_events", "Anzahl Ereignisse des jeweiligen Subtyps im Monat."),
    ],
)
display(flood_subtype_long_again.query("n_flood_events > 0").head(10))

Flood-Monatsaggregation: 155 Monate mit Floods
Long -> Wide -> Long: (196, 3) -> (155, 6) -> (775, 3)

Tabelle: Monatliche Flood-Aggregation


,Spalte,Erklärung
0,year_month,Monat der Aggregation.
1,flood_events,Anzahl eindeutiger Flood-Ereignisse pro Monat.
2,flood_countries,Anzahl betroffener Länder pro Monat.
3,flood_affected_total,Summe betroffener Personen pro Monat.
4,flood_deaths_total,Summe der Todesfälle pro Monat.
5,flood_median_duration_days,Median der Ereignisdauer in Tagen.
6,flood_coordinates_share,Anteil Ereignisse mit vorhandenen Koordinaten.
7,flood_primary_subtype,Häufigster Flood-Subtyp im Monat.
8,n_flood_subtype_*,Anzahl Ereignisse je Flood-Subtyp als Wide-Fea...
9,affected_per_flood,Durchschnittlich betroffene Personen pro Flood...


,year_month,flood_events,flood_countries,flood_affected_total,flood_deaths_total,flood_median_duration_days,flood_coordinates_share,flood_primary_subtype,n_flood_subtype_coastal_flood,n_flood_subtype_flash_flood,n_flood_subtype_flood,n_flood_subtype_glacial_lake_outburst_flood,n_flood_subtype_riverine_flood,affected_per_flood,flood_death_share
0,1999-11-01,1,1,3005.0,36.0,4.0,0.0,riverine_flood,0,0,0,0,1,3005.0,0.011980
1,2000-05-01,1,1,1000.0,2.0,1.0,0.0,riverine_flood,0,0,0,0,1,1000.0,0.002000
2,2000-06-01,2,2,700.0,17.0,2.0,0.0,flash_flood,0,1,0,0,1,350.0,0.024286
3,2000-07-01,1,1,600.0,1.0,4.0,0.0,riverine_flood,0,0,0,0,1,600.0,0.001667
4,2000-09-01,2,1,1722.0,13.0,2.0,0.0,flood,0,0,1,0,1,861.0,0.007549



Tabelle: Floods nach Land und Jahr


,Spalte,Erklärung
0,flood_year,Jahr des Flood-Ereignisses.
1,country_clean,Bereinigter Ländername.
2,flood_events,Anzahl eindeutiger Flood-Ereignisse pro Land u...
3,affected_total,Summe betroffener Personen pro Land und Jahr.
4,deaths_total,Summe Todesfälle pro Land und Jahr.


,flood_year,country_clean,flood_events,affected_total,deaths_total
0,1999,France,1,3005.0,36.0
2,2000,France,5,2302.0,3.0
4,2000,Italy,4,50322.0,44.0
6,2000,Spain,2,8500.0,26.0
3,2000,Greece,2,6600.0,1.0
5,2000,Morocco,2,950.0,6.0
7,2000,Türkiye,1,1000.0,2.0
1,2000,Algeria,1,100.0,28.0
11,2001,France,5,16882.0,3.0
14,2001,Türkiye,3,2520.0,12.0



Tabelle: Flood-Subtypen wieder im Long-Format


,Spalte,Erklärung
0,year_month,Monat der Aggregation.
1,flood_subtype,Bereinigter Flood-Subtyp nach Rücktransformati...
2,n_flood_events,Anzahl Ereignisse des jeweiligen Subtyps im Mo...


,year_month,flood_subtype,n_flood_events
7,2000-12-01,coastal_flood,1.0
52,2007-08-01,coastal_flood,1.0
157,2000-06-01,flash_flood,1.0
160,2000-10-01,flash_flood,3.0
161,2000-11-01,flash_flood,2.0
163,2001-01-01,flash_flood,2.0
170,2001-11-01,flash_flood,1.0
173,2002-06-01,flash_flood,1.0
175,2002-08-01,flash_flood,1.0
176,2002-09-01,flash_flood,1.0


### EMDAT Floods: Date Range, Lag/Lead, Moving Window und Reskalierung

Dieser Abschnitt macht aus den beobachteten Flood-Monaten eine vollständige monatliche Zeitreihe. Dadurch werden auch Monate ohne Flood-Ereignisse sichtbar und können mit 0 aufgefüllt werden.

**Input:** `flood_monthly` aus der monatlichen Aggregation.

**Was gemacht wird:** Eine vollständige Monatsrange wird mit `pd.date_range()` aufgebaut, fehlende Monate werden ergänzt, Lag-/Lead-Variablen werden mit `shift()` gebildet und rollierende 3- bzw. 12-Monatsfenster berechnet. Danach werden ausgewählte Variablen standardisiert, min-max-skaliert sowie mit Box-Cox und Yeo-Johnson transformiert.

**Output:** `flood_monthly_ts` als vollständige Zeitreihe und `flood_scaled` als skalierte Feature-Tabelle.


In [16]:
observed_flood_months = pd.DatetimeIndex(flood_monthly["year_month"].sort_values().unique())
expected_flood_months = pd.date_range(
    observed_flood_months.min(),
    observed_flood_months.max(),
    freq="MS",
)
missing_flood_months = expected_flood_months.difference(observed_flood_months)

flood_monthly_ts = (
    pd.DataFrame({"year_month": expected_flood_months})
    .merge(flood_monthly, on="year_month", how="left")
    .sort_values("year_month")
    .reset_index(drop=True)
)

flood_fill_cols = [
    "flood_events", "flood_countries", "flood_affected_total", "flood_deaths_total",
    "flood_median_duration_days", "flood_coordinates_share", "affected_per_flood", "flood_death_share",
] + [c for c in flood_monthly_ts.columns if c.startswith("n_flood_subtype_")]
for col in flood_fill_cols:
    if col in flood_monthly_ts.columns:
        flood_monthly_ts[col] = flood_monthly_ts[col].fillna(0)

flood_monthly_ts["flood_primary_subtype"] = flood_monthly_ts["flood_primary_subtype"].fillna("none")
flood_monthly_ts["has_flood"] = flood_monthly_ts["flood_events"].gt(0).astype(int)
flood_monthly_ts["flood_events_lag_1m"] = flood_monthly_ts["flood_events"].shift(1)
flood_monthly_ts["flood_events_lead_1m"] = flood_monthly_ts["flood_events"].shift(-1)
flood_monthly_ts["flood_affected_lag_1m"] = flood_monthly_ts["flood_affected_total"].shift(1)
flood_monthly_ts["flood_affected_lead_1m"] = flood_monthly_ts["flood_affected_total"].shift(-1)
flood_monthly_ts["flood_events_roll_3m_mean"] = flood_monthly_ts["flood_events"].rolling(window=3, min_periods=1).mean()
flood_monthly_ts["flood_events_roll_12m_sum"] = flood_monthly_ts["flood_events"].rolling(window=12, min_periods=1).sum()
flood_monthly_ts["flood_affected_roll_12m_sum"] = flood_monthly_ts["flood_affected_total"].rolling(window=12, min_periods=1).sum()

flood_scaled = flood_monthly_ts[["year_month", "flood_events", "flood_affected_total"]].copy()
flood_scaled["flood_events_z"] = standard_scale(flood_scaled["flood_events"])
flood_scaled["flood_events_minmax"] = minmax_scale(flood_scaled["flood_events"])
flood_scaled["flood_affected_boxcox_z"] = standard_scale(
    box_cox_transform(flood_scaled["flood_affected_total"], lmbda=0.0)
)
flood_scaled["flood_affected_yeojohnson_z"] = standard_scale(
    yeo_johnson_transform(flood_scaled["flood_affected_total"], lmbda=0.5)
)

print(f"Beobachtete Flood-Monate: {len(observed_flood_months):,}")
print(f"Erwartete Flood-Monate im ausgerichteten Zeitraum: {len(expected_flood_months):,}")
print(f"Fehlende Monate in der Flood-Zeitreihe: {len(missing_flood_months):,}")
show_table_info(
    "Tabelle: Vollständige monatliche Flood-Zeitreihe",
    [
        ("year_month", "Monat in der vollständigen Date Range."),
        ("flood_events", "Anzahl Flood-Ereignisse im Monat, fehlende Monate mit 0 ergänzt."),
        ("flood_events_lag_1m", "Flood-Ereignisse des Vormonats."),
        ("flood_events_lead_1m", "Flood-Ereignisse des Folgemonats."),
        ("flood_events_roll_3m_mean", "Rollierender 3-Monats-Mittelwert der Ereignisse."),
        ("flood_events_roll_12m_sum", "Rollierende 12-Monats-Summe der Ereignisse."),
        ("flood_affected_total", "Betroffene Personen im Monat."),
        ("flood_affected_roll_12m_sum", "Rollierende 12-Monats-Summe betroffener Personen."),
    ],
)
display(
    flood_monthly_ts[
        [
            "year_month", "flood_events", "flood_events_lag_1m", "flood_events_lead_1m",
            "flood_events_roll_3m_mean", "flood_events_roll_12m_sum", "flood_affected_total",
            "flood_affected_roll_12m_sum",
        ]
    ].head(15)
)
show_table_info(
    "Tabelle: Skalierte Flood-Variablen",
    [
        ("year_month", "Monat der Beobachtung."),
        ("flood_events", "Originale Anzahl Flood-Ereignisse."),
        ("flood_affected_total", "Originale Anzahl betroffener Personen."),
        ("flood_events_z", "Standardisierte Flood-Ereignisse mit Mittelwert 0 und Standardabweichung 1."),
        ("flood_events_minmax", "Min-Max-skalierte Flood-Ereignisse zwischen 0 und 1."),
        ("flood_affected_boxcox_z", "Box-Cox-transformierte und standardisierte Betroffenenzahl."),
        ("flood_affected_yeojohnson_z", "Yeo-Johnson-transformierte und standardisierte Betroffenenzahl."),
    ],
)
display(flood_scaled.head(10))

Beobachtete Flood-Monate: 155
Erwartete Flood-Monate im ausgerichteten Zeitraum: 301
Fehlende Monate in der Flood-Zeitreihe: 146

Tabelle: Vollständige monatliche Flood-Zeitreihe


,Spalte,Erklärung
0,year_month,Monat in der vollständigen Date Range.
1,flood_events,"Anzahl Flood-Ereignisse im Monat, fehlende Mon..."
2,flood_events_lag_1m,Flood-Ereignisse des Vormonats.
3,flood_events_lead_1m,Flood-Ereignisse des Folgemonats.
4,flood_events_roll_3m_mean,Rollierender 3-Monats-Mittelwert der Ereignisse.
5,flood_events_roll_12m_sum,Rollierende 12-Monats-Summe der Ereignisse.
6,flood_affected_total,Betroffene Personen im Monat.
7,flood_affected_roll_12m_sum,Rollierende 12-Monats-Summe betroffener Personen.


,year_month,flood_events,flood_events_lag_1m,flood_events_lead_1m,flood_events_roll_3m_mean,flood_events_roll_12m_sum,flood_affected_total,flood_affected_roll_12m_sum
0,1999-11-01,1.0,NaN,0.0,1.000000,1.0,3005.0,3005.0
1,1999-12-01,0.0,1.0,0.0,0.500000,1.0,0.0,3005.0
2,2000-01-01,0.0,0.0,0.0,0.333333,1.0,0.0,3005.0
3,2000-02-01,0.0,0.0,0.0,0.000000,1.0,0.0,3005.0
4,2000-03-01,0.0,0.0,0.0,0.000000,1.0,0.0,3005.0
5,2000-04-01,0.0,0.0,1.0,0.000000,1.0,0.0,3005.0
6,2000-05-01,1.0,0.0,2.0,0.333333,2.0,1000.0,4005.0
7,2000-06-01,2.0,1.0,1.0,1.000000,4.0,700.0,4705.0
8,2000-07-01,1.0,2.0,0.0,1.333333,5.0,600.0,5305.0
9,2000-08-01,0.0,1.0,2.0,1.000000,5.0,0.0,5305.0



Tabelle: Skalierte Flood-Variablen


,Spalte,Erklärung
0,year_month,Monat der Beobachtung.
1,flood_events,Originale Anzahl Flood-Ereignisse.
2,flood_affected_total,Originale Anzahl betroffener Personen.
3,flood_events_z,Standardisierte Flood-Ereignisse mit Mittelwer...
4,flood_events_minmax,Min-Max-skalierte Flood-Ereignisse zwischen 0 ...
5,flood_affected_boxcox_z,Box-Cox-transformierte und standardisierte Bet...
6,flood_affected_yeojohnson_z,Yeo-Johnson-transformierte und standardisierte...


,year_month,flood_events,flood_affected_total,flood_events_z,flood_events_minmax,flood_affected_boxcox_z,flood_affected_yeojohnson_z
0,1999-11-01,1.0,3005.0,0.024220,0.142857,0.997812,0.057970
1,1999-12-01,0.0,0.0,-0.785816,0.000000,-0.886093,-0.399504
2,2000-01-01,0.0,0.0,-0.785816,0.000000,-0.886093,-0.399504
3,2000-02-01,0.0,0.0,-0.785816,0.000000,-0.886093,-0.399504
4,2000-03-01,0.0,0.0,-0.785816,0.000000,-0.886093,-0.399504
5,2000-04-01,0.0,0.0,-0.785816,0.000000,-0.886093,-0.399504
6,2000-05-01,1.0,1000.0,0.024220,0.142857,0.739137,-0.139107
7,2000-06-01,2.0,700.0,0.834256,0.285714,0.655333,-0.182981
8,2000-07-01,1.0,600.0,0.024220,0.142857,0.619126,-0.199648
9,2000-08-01,0.0,0.0,-0.785816,0.000000,-0.886093,-0.399504


### Sea-Level separat transformieren

Der Sea-Level-Datensatz wird unabhängig von EMDAT vorbereitet. Dadurch bleibt klar getrennt, welche Transformationen zur Klimazeitreihe gehören und welche zu den Katastrophendaten.

**Input:** `sea_level` nach `run_transform()` und `run_cleaning()`.

**Was gemacht wird:** Die Tageswerte werden sortiert, mit Monatsinformationen ergänzt, in Long- und Wide-Formate umgewandelt, monatlich aggregiert und mit Lag-/Lead- sowie Rolling-Window-Features versehen. Zusätzlich werden Sea-Level-Variablen skaliert.

**Output:** `sea_level_ts`, `sea_metric_long`, `sea_metric_wide_again`, `sea_monthly` und `sea_scaled`.


In [17]:
sea_level_ts = sea_level.copy()
sea_level_ts["time"] = pd.to_datetime(sea_level_ts["time"])
sea_level_ts = sea_level_ts.sort_values("time").reset_index(drop=True)
sea_level_ts["year_month"] = sea_level_ts["time"].dt.to_period("M").dt.to_timestamp()
sea_level_ts["year"] = sea_level_ts["time"].dt.year
sea_level_ts["month"] = sea_level_ts["time"].dt.month
sea_level_ts["msl_anomaly_mm"] = sea_level_ts[SEA_VALUE] * 10
sea_level_ts["msl_minus_trend_cm"] = sea_level_ts[SEA_VALUE] - sea_level_ts[SEA_TREND]

sea_metric_long = sea_level_ts.melt(
    id_vars=["time", "year_month"],
    value_vars=[SEA_VALUE, SEA_TREND, "msl_anomaly_mm", "msl_minus_trend_cm"],
    var_name="sea_metric",
    value_name="sea_value",
)
sea_metric_wide_again = (
    sea_metric_long.pivot_table(
        index=["time", "year_month"],
        columns="sea_metric",
        values="sea_value",
    )
    .reset_index()
)
sea_metric_wide_again.columns.name = None

sea_monthly = (
    sea_level_ts.set_index("time")
    .resample("MS")
    .agg(
        msl_mean_cm=(SEA_VALUE, "mean"),
        msl_min_cm=(SEA_VALUE, "min"),
        msl_max_cm=(SEA_VALUE, "max"),
        trend_mean_cm=(SEA_TREND, "mean"),
        n_days=(SEA_VALUE, "size"),
    )
    .rename_axis("year_month")
    .reset_index()
)
sea_monthly["msl_range_cm"] = sea_monthly["msl_max_cm"] - sea_monthly["msl_min_cm"]
sea_monthly["msl_mean_lag_1m"] = sea_monthly["msl_mean_cm"].shift(1)
sea_monthly["msl_mean_lead_1m"] = sea_monthly["msl_mean_cm"].shift(-1)
sea_monthly["msl_mean_roll_3m"] = sea_monthly["msl_mean_cm"].rolling(window=3, min_periods=1).mean()
sea_monthly["msl_mean_roll_12m"] = sea_monthly["msl_mean_cm"].rolling(window=12, min_periods=1).mean()

sea_month_range = pd.date_range(sea_monthly["year_month"].min(), sea_monthly["year_month"].max(), freq="MS")
missing_sea_months = sea_month_range.difference(sea_monthly["year_month"])

sea_scaled = sea_monthly[["year_month", "msl_mean_cm", "msl_range_cm"]].copy()
sea_scaled["msl_mean_cm_z"] = standard_scale(sea_scaled["msl_mean_cm"])
sea_scaled["msl_mean_cm_minmax"] = minmax_scale(sea_scaled["msl_mean_cm"])
sea_scaled["msl_range_boxcox_z"] = standard_scale(box_cox_transform(sea_scaled["msl_range_cm"], lmbda=0.0))
sea_scaled["msl_mean_yeojohnson_z"] = standard_scale(yeo_johnson_transform(sea_scaled["msl_mean_cm"], lmbda=0.5))

print(f"Sea-Level Tage: {len(sea_level_ts):,}")
print(f"Sea-Level Monate: {len(sea_monthly):,}; fehlende Monate: {len(missing_sea_months):,}")
print(f"Sea-Level Wide -> Long -> Wide: {sea_level_ts[[SEA_VALUE, SEA_TREND]].shape} -> {sea_metric_long.shape} -> {sea_metric_wide_again.shape}")
show_table_info(
    "Tabelle: Monatliche Sea-Level-Aggregation",
    [
        ("year_month", "Monat der Aggregation."),
        ("msl_mean_cm", "Monatlicher Mittelwert der Meeresspiegelanomalie in cm."),
        ("msl_min_cm", "Monatliches Minimum der Meeresspiegelanomalie in cm."),
        ("msl_max_cm", "Monatliches Maximum der Meeresspiegelanomalie in cm."),
        ("trend_mean_cm", "Monatlicher Mittelwert des Trendwerts in cm."),
        ("n_days", "Anzahl Tageswerte im Monat."),
        ("msl_range_cm", "Spannweite zwischen Monatsmaximum und Monatsminimum."),
        ("msl_mean_lag_1m", "Sea-Level-Mittelwert des Vormonats."),
        ("msl_mean_lead_1m", "Sea-Level-Mittelwert des Folgemonats."),
        ("msl_mean_roll_3m", "Rollierender 3-Monats-Mittelwert."),
        ("msl_mean_roll_12m", "Rollierender 12-Monats-Mittelwert."),
    ],
)
display(sea_monthly.head())
show_table_info(
    "Tabelle: Sea-Level-Metriken im Long-Format",
    [
        ("time", "Ursprünglicher Tageszeitpunkt."),
        ("year_month", "Zugehöriger Monat."),
        ("sea_metric", "Name der jeweiligen Sea-Level-Metrik."),
        ("sea_value", "Wert der jeweiligen Metrik."),
    ],
)
display(sea_metric_long.head(8))
show_table_info(
    "Tabelle: Skalierte Sea-Level-Variablen",
    [
        ("year_month", "Monat der Beobachtung."),
        ("msl_mean_cm", "Originaler monatlicher Sea-Level-Mittelwert in cm."),
        ("msl_range_cm", "Originale monatliche Sea-Level-Spannweite in cm."),
        ("msl_mean_cm_z", "Standardisierter Sea-Level-Mittelwert."),
        ("msl_mean_cm_minmax", "Min-Max-skalierter Sea-Level-Mittelwert."),
        ("msl_range_boxcox_z", "Box-Cox-transformierte und standardisierte Sea-Level-Spannweite."),
        ("msl_mean_yeojohnson_z", "Yeo-Johnson-transformierter und standardisierter Sea-Level-Mittelwert."),
    ],
)
display(sea_scaled.head(10))

Sea-Level Tage: 9,405
Sea-Level Monate: 310; fehlende Monate: 0
Sea-Level Wide -> Long -> Wide: (9405, 2) -> (37620, 4) -> (9405, 6)

Tabelle: Monatliche Sea-Level-Aggregation


,Spalte,Erklärung
0,year_month,Monat der Aggregation.
1,msl_mean_cm,Monatlicher Mittelwert der Meeresspiegelanomal...
2,msl_min_cm,Monatliches Minimum der Meeresspiegelanomalie ...
3,msl_max_cm,Monatliches Maximum der Meeresspiegelanomalie ...
4,trend_mean_cm,Monatlicher Mittelwert des Trendwerts in cm.
5,n_days,Anzahl Tageswerte im Monat.
6,msl_range_cm,Spannweite zwischen Monatsmaximum und Monatsmi...
7,msl_mean_lag_1m,Sea-Level-Mittelwert des Vormonats.
8,msl_mean_lead_1m,Sea-Level-Mittelwert des Folgemonats.
9,msl_mean_roll_3m,Rollierender 3-Monats-Mittelwert.


,year_month,msl_mean_cm,msl_min_cm,msl_max_cm,trend_mean_cm,n_days,msl_range_cm,msl_mean_lag_1m,msl_mean_lead_1m,msl_mean_roll_3m,msl_mean_roll_12m
0,1999-02-01,3.672906,3.654833,3.687602,2.173262,9,0.032769,NaN,3.507500,3.672906,3.672906
1,1999-03-01,3.507500,3.317693,3.649111,2.182188,31,0.331417,3.672906,3.030831,3.590203,3.590203
2,1999-04-01,3.030831,2.732398,3.301683,2.195883,30,0.569285,3.507500,2.385838,3.403746,3.403746
3,1999-05-01,2.385838,2.083335,2.710393,2.209686,31,0.627059,3.030831,1.898927,2.974723,3.149269
4,1999-06-01,1.898927,1.817051,2.066172,2.223596,30,0.249121,2.385838,1.939131,2.438532,2.899200



Tabelle: Sea-Level-Metriken im Long-Format


,Spalte,Erklärung
0,time,Ursprünglicher Tageszeitpunkt.
1,year_month,Zugehöriger Monat.
2,sea_metric,Name der jeweiligen Sea-Level-Metrik.
3,sea_value,Wert der jeweiligen Metrik.


,time,year_month,sea_metric,sea_value
0,1999-02-20,1999-02-01,MSL_filtered_GIA_corrected_adjusted,3.687602
1,1999-02-21,1999-02-01,MSL_filtered_GIA_corrected_adjusted,3.684773
2,1999-02-22,1999-02-01,MSL_filtered_GIA_corrected_adjusted,3.681582
3,1999-02-23,1999-02-01,MSL_filtered_GIA_corrected_adjusted,3.678028
4,1999-02-24,1999-02-01,MSL_filtered_GIA_corrected_adjusted,3.674113
5,1999-02-25,1999-02-01,MSL_filtered_GIA_corrected_adjusted,3.669835
6,1999-02-26,1999-02-01,MSL_filtered_GIA_corrected_adjusted,3.665195
7,1999-02-27,1999-02-01,MSL_filtered_GIA_corrected_adjusted,3.660195



Tabelle: Skalierte Sea-Level-Variablen


,Spalte,Erklärung
0,year_month,Monat der Beobachtung.
1,msl_mean_cm,Originaler monatlicher Sea-Level-Mittelwert in...
2,msl_range_cm,Originale monatliche Sea-Level-Spannweite in cm.
3,msl_mean_cm_z,Standardisierter Sea-Level-Mittelwert.
4,msl_mean_cm_minmax,Min-Max-skalierter Sea-Level-Mittelwert.
5,msl_range_boxcox_z,Box-Cox-transformierte und standardisierte Sea...
6,msl_mean_yeojohnson_z,Yeo-Johnson-transformierter und standardisiert...


,year_month,msl_mean_cm,msl_range_cm,msl_mean_cm_z,msl_mean_cm_minmax,msl_range_boxcox_z,msl_mean_yeojohnson_z
0,1999-02-01,3.672906,0.032769,-0.697887,0.271752,-1.538154,-0.612841
1,1999-03-01,3.507500,0.331417,-0.748258,0.260624,-0.677431,-0.672472
2,1999-04-01,3.030831,0.569285,-0.893417,0.228556,-0.121385,-0.850723
3,1999-05-01,2.385838,0.627059,-1.089836,0.185164,0.000827,-1.109659
4,1999-06-01,1.898927,0.249121,-1.238114,0.152407,-0.893456,-1.321966
5,1999-07-01,1.939131,0.310075,-1.225871,0.155112,-0.732132,-1.303791
6,1999-08-01,2.373732,0.411141,-1.093523,0.184350,-0.480629,-1.114745
7,1999-09-01,2.546864,0.168565,-1.040799,0.195997,-1.119292,-1.042855
8,1999-10-01,2.068530,0.738520,-1.186465,0.163817,0.224735,-1.246123
9,1999-11-01,1.229291,0.767820,-1.442037,0.107357,0.281197,-1.645647


# Daten Joinen

# Entscheidung Long oder wide Format

### Logs in Funktionen

In diesem Abschnitt wird gezeigt, wie Transformationslogik in Funktionen mit Logging gekapselt wird. Die Logs dokumentieren Start, Ende und mögliche Fehler der Aggregationen.

**Input:** `emdat_flood` und `sea_level_ts` aus den vorherigen Abschnitten.

**Was gemacht wird:** Zwei Funktionen prüfen zuerst, ob alle notwendigen Spalten vorhanden sind. Bei fehlenden Spalten wird ein Fehler geloggt und eine Exception ausgelöst. Wenn alles passt, wird aggregiert und der erfolgreiche Abschluss geloggt.

**Output:** `flood_monthly_logged` und `sea_monthly_logged` sowie sichtbare Logmeldungen im Notebook-Output.


In [18]:
logger = logging.getLogger("myproj.transform.flood")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False


def build_flood_monthly_with_logs(flood_events):
    logger.info("Starte Flood-Aggregation: %s Ereignisse", len(flood_events))
    required_cols = {"year_month", "DisNo.", "affected_total_filled", "deaths_total_filled", "flood_subtype_clean"}
    missing_cols = required_cols.difference(flood_events.columns)
    if missing_cols:
        logger.error("Fehlende Flood-Spalten: %s", sorted(missing_cols))
        raise ValueError(f"Fehlende Flood-Spalten: {sorted(missing_cols)}")

    result = (
        flood_events.groupby("year_month")
        .agg(
            flood_events=("DisNo.", "nunique"),
            flood_affected_total=("affected_total_filled", "sum"),
            flood_deaths_total=("deaths_total_filled", "sum"),
            flood_primary_subtype=("flood_subtype_clean", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        )
        .reset_index()
    )
    logger.info("Flood-Aggregation fertig: %s Monate", len(result))
    return result


def build_sea_monthly_with_logs(sea_daily):
    logger.info("Starte Sea-Level-Aggregation: %s Tageswerte", len(sea_daily))
    required_cols = {"time", SEA_VALUE, SEA_TREND}
    missing_cols = required_cols.difference(sea_daily.columns)
    if missing_cols:
        logger.error("Fehlende Sea-Level-Spalten: %s", sorted(missing_cols))
        raise ValueError(f"Fehlende Sea-Level-Spalten: {sorted(missing_cols)}")

    result = (
        sea_daily.set_index("time")
        .resample("MS")
        .agg(msl_mean_cm=(SEA_VALUE, "mean"), trend_mean_cm=(SEA_TREND, "mean"))
        .rename_axis("year_month")
        .reset_index()
    )
    logger.info("Sea-Level-Aggregation fertig: %s Monate", len(result))
    return result


flood_monthly_logged = build_flood_monthly_with_logs(emdat_flood)
sea_monthly_logged = build_sea_monthly_with_logs(sea_level_ts)

show_table_info(
    "Tabelle: Geloggte Flood-Aggregation",
    [
        ("year_month", "Monat der Aggregation."),
        ("flood_events", "Anzahl eindeutiger Flood-Ereignisse."),
        ("flood_affected_total", "Summe betroffener Personen."),
        ("flood_deaths_total", "Summe Todesfälle."),
        ("flood_primary_subtype", "Häufigster Flood-Subtyp im Monat."),
    ],
)
display(flood_monthly_logged.head())
show_table_info(
    "Tabelle: Geloggte Sea-Level-Aggregation",
    [
        ("year_month", "Monat der Aggregation."),
        ("msl_mean_cm", "Monatlicher Mittelwert der Meeresspiegelanomalie in cm."),
        ("trend_mean_cm", "Monatlicher Mittelwert des Sea-Level-Trends in cm."),
    ],
)
display(sea_monthly_logged.head())

INFO:myproj.transform.flood:Starte Flood-Aggregation: 292 Ereignisse
INFO:myproj.transform.flood:Flood-Aggregation fertig: 155 Monate
INFO:myproj.transform.flood:Starte Sea-Level-Aggregation: 9405 Tageswerte
INFO:myproj.transform.flood:Sea-Level-Aggregation fertig: 310 Monate



Tabelle: Geloggte Flood-Aggregation


,Spalte,Erklärung
0,year_month,Monat der Aggregation.
1,flood_events,Anzahl eindeutiger Flood-Ereignisse.
2,flood_affected_total,Summe betroffener Personen.
3,flood_deaths_total,Summe Todesfälle.
4,flood_primary_subtype,Häufigster Flood-Subtyp im Monat.


,year_month,flood_events,flood_affected_total,flood_deaths_total,flood_primary_subtype
0,1999-11-01,1,3005.0,36.0,riverine_flood
1,2000-05-01,1,1000.0,2.0,riverine_flood
2,2000-06-01,2,700.0,17.0,flash_flood
3,2000-07-01,1,600.0,1.0,riverine_flood
4,2000-09-01,2,1722.0,13.0,flood



Tabelle: Geloggte Sea-Level-Aggregation


,Spalte,Erklärung
0,year_month,Monat der Aggregation.
1,msl_mean_cm,Monatlicher Mittelwert der Meeresspiegelanomal...
2,trend_mean_cm,Monatlicher Mittelwert des Sea-Level-Trends in...


,year_month,msl_mean_cm,trend_mean_cm
0,1999-02-01,3.672906,2.173262
1,1999-03-01,3.507500,2.182188
2,1999-04-01,3.030831,2.195883
3,1999-05-01,2.385838,2.209686
4,1999-06-01,1.898927,2.223596
